In [130]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('sci.mplstyle')

In [131]:
# Tham so dau vo
N = 100
epsilon_max = 300 #(meV)
delta_epsilon = epsilon_max / N

a_0 = 125 #(angstrom)
E_R = 4.2 #meV
hbar = 658.5 #meV*fs
T_2 = 200 #fs

# Tham so laser
chi_0 = 0.1
delta_t = 25 #fs (be rong cua xung laser, 25 la mac dinh)
Delta_0 = 30 #meV #30 la mac dinh
dt = 2 #fs
t_max = 500 #fs
t_0 = -3 * delta_t #fs (thoi diem xung laser dat gia tri cuc dai)

output_name = 'SBE_thamso_chi_macdinh'

In [132]:
# Tham so dau vo
N = 100
epsilon_max = 300 #(meV)
delta_epsilon = epsilon_max / N

a_0 = 125 #(angstrom)
E_R = 4.2 #meV
hbar = 658.5 #meV*fs
T_2 = 200 #fs

# Tham so laser
chi_0 = 0.1
delta_t = 50 #fs (be rong cua xung laser, 25 la mac dinh)
Delta_0 = 100 #meV #(detuning, 30 la mac dinh)
dt = 2 #fs
t_max = 500 #fs
t_0 = -3 * delta_t #fs (thoi diem xung laser dat gia tri cuc dai)

output_name = 'SBE_thamso_chi_delta0_100_deltat_50'

In [133]:
def khoitao_g_matrix():
    global N, delta_epsilon, E_R, g_matrix
    g_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            i_khac0 = i + 1
            j_khac0 = j + 1
            if i_khac0 == j_khac0:
                g_matrix[i][j] = 0
            else:
                g_matrix[i][j] = 1/np.sqrt(i_khac0*delta_epsilon) * np.log(np.abs((np.sqrt(i_khac0)+np.sqrt(j_khac0))/(np.sqrt(i_khac0)-np.sqrt(j_khac0))))
    return g_matrix

def cal_E_tinhtong(Y):
    global N, delta_epsilon, E_R, g_matrix
    E = np.zeros(N)
    for n in range(0,N):
        phan_tong = 0
        for n1 in range(0,N):
            phan_tong = phan_tong + g_matrix[n][n1]*(Y[0][n1].real + Y[0][n1].imag)
        E[n] = phan_tong*(np.sqrt(E_R)*delta_epsilon/np.pi)
    return E

def cal_E_dung_nhan_matran(Y):
    global N, delta_epsilon, E_R, g_matrix
    tong_mat_do_hat = Y[0].real + Y[0].imag
    tinh_tich_chap = np.dot(g_matrix, tong_mat_do_hat)
    E = tinh_tich_chap*(np.sqrt(E_R)*delta_epsilon/np.pi)
    return E

def Omega(Y, t):
    global N, delta_epsilon, E_R, chi_0, Delta_0, delta_t, hbar
    nhan_matran_g_va_p = np.dot(g_matrix, Y[1])
    Omega  = 1/hbar * (1/2 * hbar * np.sqrt(np.pi)/delta_t * chi_0 * np.exp(-((t)/delta_t)**2) + np.sqrt(E_R)*delta_epsilon/np.pi * nhan_matran_g_va_p)
    return Omega

def calculate_F(t,Y):
    global N, delta_epsilon, E_R, chi_0, Delta_0, delta_t, hbar, T_2
    F = np.zeros((2,N), dtype=np.complex128)
    E = cal_E_dung_nhan_matran(Y)
    Omega_arr = Omega(Y,t)
    a = np.zeros(N)
    for n in range(N):
        n_cong_thuc = n + 1
        a[n] = (-2* Omega_arr[n]*(Y[1][n]).conjugate()).imag
        
        F[0][n] = a[n] + 1j*a[n]
        F[1][n] = -1j/hbar *(n_cong_thuc*delta_epsilon - Delta_0 - E[n])*Y[1][n] + 1j*(1-Y[0][n].real - Y[0][n].imag)*Omega_arr[n] - Y[1][n]/T_2
    return F

In [134]:
def luufileketqua_taofile(output_name):
    global N, epsilon_max, delta_epsilon, a_0, E_R, hbar, T_2, chi_0, delta_t, Delta_0, dt, t_max
    
    filename = f'SBE-ketqua-{output_name}.txt'
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write('#' * 100 + '\n')
        f.write('# KET QUA GIAI PHUONG TRINH SEMICONDUCTOR BLOCH EQUATIONS (SBE)\n')
        f.write('#' * 100 + '\n\n')
        
        f.write('# ====================================================================\n')
        f.write('# THAM SO DAU VAO\n')
        f.write('# ====================================================================\n\n')
        
        f.write('# --- THAM SO TINH TOAN ---\n')
        f.write(f'# N (So diem luoi nang luong)  = {N}\n')
        f.write(f'# epsilon_max (Nang luong max) = {epsilon_max} meV\n')
        f.write(f'# delta_epsilon (Buoc luoi)    = {delta_epsilon} meV\n')
        f.write(f'# dt (Buoc thoi gian)          = {dt} fs\n')
        f.write(f'# t_max (Thoi gian mo phong)   = {t_max} fs\n')
        f.write('#\n')
        
        f.write('# --- THAM SO VAT LY HE BAN DAN ---\n')
        f.write(f'# a_0 (Ban kinh Bohr Exciton)  = {a_0} angstrom\n')
        f.write(f'# E_R (Nang luong Rydberg)     = {E_R} meV\n')
        f.write(f'# T_2 (Thoi gian dephasing)    = {T_2} fs\n')
        f.write(f'# hbar (Hang so Planck thu gon) = {hbar} meV*fs\n')
        f.write('#\n')
        
        f.write('# --- THAM SO KICH THICH LASER ---\n')
        f.write(f'# chi_0 (Tan so Rabi dinh)     = {chi_0}\n')
        f.write(f'# Delta_0 (Do lech pha/Detuning)= {Delta_0} meV\n')
        f.write(f'# delta_t (Do rong xung laser) = {delta_t} fs\n')
        
        f.write('\n' + '#' * 100 + '\n\n')
        
        f.write(f"#{'n':>5s}  {'t':>15s}  {'Epsilon':>15s}  {'Y0_real (f_e)':>15s}  {'Y0_imag (f_h)':>15s}  {'Y1_real':>15s}  {'Y1_imag':>15s}\n")


def luufileketqua_luuketqua(Y, t, output_name):
    global N, delta_epsilon
    filename = f'SBE-ketqua-{output_name}.txt'
    N_e = 0.0
    N_h = 0.0
    P = 0.0
    with open(filename, 'a', encoding='utf-8') as f:
        lines = []
        for n in range(N):
            epsilon_n = (n + 1) * delta_epsilon
            Y0_real = Y[0][n].real
            Y0_imag = Y[0][n].imag
            Y1_real = Y[1][n].real
            Y1_imag = Y[1][n].imag
            N_e = N_e + np.sqrt(n+1)*Y0_real
            N_h = N_h + np.sqrt(n+1)*Y0_imag
            P  = P + np.sqrt(n+1)*np.abs(Y[1][n])
            line = (f" {n:>5d}  "
                    f"{t:>15.6e}  "
                    f"{epsilon_n:>15.6e}  "
                    f"{Y0_real:>15.6e}  "
                    f"{Y0_imag:>15.6e}  "
                    f"{Y1_real:>15.6e}  "
                    f"{Y1_imag:>15.6e}\n")
            lines.append(line)
        
        f.writelines(lines)
        f.write('\n')
        return N_e, N_h, P

In [135]:
### File tinh N va P theo thoi gian
def luufileketqua_taofile_N_va_P(output_name):
    global N, epsilon_max, delta_epsilon, a_0, E_R, hbar, T_2, chi_0, delta_t, Delta_0, dt, t_max
    
    filename = f'SBE-ketqua-{output_name}-N-P.txt'
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write('#' * 100 + '\n')
        f.write('# KET QUA GIAI PHUONG TRINH SEMICONDUCTOR BLOCH EQUATIONS (SBE)\n')
        f.write('#' * 100 + '\n\n')
        
        f.write('# ====================================================================\n')
        f.write('# THAM SO DAU VAO\n')
        f.write('# ====================================================================\n\n')
        
        f.write('# --- THAM SO TINH TOAN ---\n')
        f.write(f'# N (So diem luoi nang luong)  = {N}\n')
        f.write(f'# epsilon_max (Nang luong max) = {epsilon_max} meV\n')
        f.write(f'# delta_epsilon (Buoc luoi)    = {delta_epsilon} meV\n')
        f.write(f'# dt (Buoc thoi gian)          = {dt} fs\n')
        f.write(f'# t_max (Thoi gian mo phong)   = {t_max} fs\n')
        f.write('#\n')
        
        f.write('# --- THAM SO VAT LY HE BAN DAN ---\n')
        f.write(f'# a_0 (Ban kinh Bohr Exciton)  = {a_0} angstrom\n')
        f.write(f'# E_R (Nang luong Rydberg)     = {E_R} meV\n')
        f.write(f'# T_2 (Thoi gian dephasing)    = {T_2} fs\n')
        f.write(f'# hbar (Hang so Planck thu gon) = {hbar} meV*fs\n')
        f.write('#\n')
        
        f.write('# --- THAM SO KICH THICH LASER ---\n')
        f.write(f'# chi_0 (Tan so Rabi dinh)     = {chi_0}\n')
        f.write(f'# Delta_0 (Do lech pha/Detuning)= {Delta_0} meV\n')
        f.write(f'# delta_t (Do rong xung laser) = {delta_t} fs\n')
        
        f.write('\n' + '#' * 100 + '\n\n')
        
        f.write(f"#{'t':>15s} {'N_e':>15s} {'N_h':>15s} {'P':>15s} \n")


def luufileketqua_luuketqua_N_va_P(t):
    global N, delta_epsilon, N_e, N_h, P, output_name
    filename = f'SBE-ketqua-{output_name}-N-P.txt'   
    with open(filename, 'a', encoding='utf-8') as f:
        lines = []
        line = (f" {t:>15.6e}  "
                f"{N_e:>15.6e}  "
                f"{N_h:>15.6e}  "
                f"{P:>15.6e}\n")
        lines.append(line)
        f.writelines(lines)

In [136]:
def RK4_SBE(alpha, phuongtrinh_SBE, filename):
    global t_0, t_max, dt, N
    N_epsilon = N
    N_steps = int((t_max - t_0) / dt)
    t = np.linspace(t_0, t_max, N_steps + 1)
    Y = np.zeros((N_steps + 1, 2, N_epsilon), dtype=np.complex128)  # Binh thuong la cho Y = np.zeros((2, N), dtype=np.complex128) (khong co buoc thoi gian)
    Y[0] = alpha

    # Vong lap thoi gian RK4
    for i in range(1, N_steps + 1):
        t_prev = t[i-1]
        Y_prev = Y[i-1]
        k1 = dt * phuongtrinh_SBE(t_prev, Y_prev)
        k2 = dt * phuongtrinh_SBE(t_prev + dt/2, Y_prev + k1/2)
        k3 = dt * phuongtrinh_SBE(t_prev + dt/2, Y_prev + k2/2)
        k4 = dt * phuongtrinh_SBE(t_prev + dt, Y_prev + k3)

        Y[i] = Y_prev + (k1 + 2*k2 + 2*k3 + k4) / 6
    return t, Y

In [137]:
if __name__ == "__main__":
    print(">>> Dang tao ma tran tuong tac Coulumb")
    khoitao_g_matrix()
    
    print(">>> Tao file ghi ket qua")
    luufileketqua_taofile(output_name)
    luufileketqua_taofile_N_va_P(output_name)
    
    Y_init = np.zeros((2, N), dtype=np.complex128) 
    
    print(">>> Bat dau giai he phuong trinh vi phan")
    t_arr, Y_sol = RK4_SBE(Y_init, calculate_F, output_name)
    print(">>> Giai he phuong trinh vi phan da xong")
    
    print(">>> Dang luu du lieu")
    for i in range(0, len(t_arr)):
        N_e, N_h, P = luufileketqua_luuketqua(Y_sol[i], t_arr[i], output_name)
        luufileketqua_luuketqua_N_va_P(t_arr[i])
        
    print(f"\n>>> Qua trinh giai da xong: SBE-ketqua-{output_name}.txt")

>>> Dang tao ma tran tuong tac Coulumb
>>> Tao file ghi ket qua
>>> Bat dau giai he phuong trinh vi phan
>>> Giai he phuong trinh vi phan da xong
>>> Dang luu du lieu

>>> Qua trinh giai da xong: SBE-ketqua-SBE_thamso_chi_delta0_100_deltat_50.txt
